# Freeze the validation evaluation protocol
Attach the private derived dataset `thestonedape/task-aware-eegtotext`, enable Internet, and enable the private `GITHUB_TOKEN` secret. This CPU-only notebook creates deterministic validation candidate pools, wrong-real-EEG donor maps, task-gated feature masks, and a contract report. It cannot access the held-out test split.

In [ ]:
REPO_URL = 'https://github.com/thestonedape/task-aware-eeg2text.git'
COMMIT = 'f3a656d59df7b119231275ae23da1063c549933f'
WORKTREE = '/kaggle/working/SemKey'
OUTPUT = '/kaggle/working/task-aware-eeg2text-evaluation-protocol'
PHASE = 'val'
POOL_SIZE = 24
SEED = 20260716
assert len(COMMIT) == 40 and all(c in '0123456789abcdef' for c in COMMIT)
assert PHASE == 'val', 'Development protocol must not access held-out test'

In [ ]:
import glob, hashlib, json, os, platform, shutil, subprocess, sys
from kaggle_secrets import UserSecretsClient
print({'python': platform.python_version(), 'platform': platform.platform()})
github_token = UserSecretsClient().get_secret('GITHUB_TOKEN')
assert github_token, 'Enable the private Kaggle Secret named GITHUB_TOKEN'
askpass = '/kaggle/working/git_askpass.py'
with open(askpass, 'w', encoding='utf-8') as handle:
    handle.write("#!/usr/bin/env python3\nimport os, sys\nprompt = sys.argv[1] if len(sys.argv) > 1 else ''\nprint(os.environ['GITHUB_TOKEN'] if 'Password' in prompt else 'x-access-token')\n")
os.chmod(askpass, 0o700)
clone_env = os.environ.copy()
clone_env.update({'GIT_ASKPASS': askpass, 'GIT_TERMINAL_PROMPT': '0', 'GITHUB_TOKEN': github_token})
if os.path.exists(WORKTREE):
    shutil.rmtree(WORKTREE)
try:
    subprocess.run(['git', 'clone', REPO_URL, WORKTREE], check=True, env=clone_env)
finally:
    os.remove(askpass)
    del github_token, clone_env
subprocess.run(['git', '-C', WORKTREE, 'checkout', '--detach', COMMIT], check=True)
actual_commit = subprocess.check_output(['git', '-C', WORKTREE, 'rev-parse', 'HEAD'], text=True).strip()
assert actual_commit == COMMIT
subprocess.run([sys.executable, os.path.join(WORKTREE, 'evaluation', 'test_protocol_manifests.py')], check=True)

In [ ]:
manifest_paths = glob.glob('/kaggle/input/**/metadata/shard_manifest.json', recursive=True)
assert len(manifest_paths) == 1, ('Attach exactly one canonical sharded dataset', manifest_paths)
dataset_root = os.path.dirname(os.path.dirname(manifest_paths[0]))
if os.path.exists(OUTPUT):
    shutil.rmtree(OUTPUT)
subprocess.run([
    sys.executable, os.path.join(WORKTREE, 'evaluation', 'build_protocol_manifests.py'),
    '--dataset-root', dataset_root, '--output-root', OUTPUT, '--phase', PHASE,
    '--pool-size', str(POOL_SIZE), '--seed', str(SEED),
], check=True)

In [ ]:
report_path = os.path.join(OUTPUT, 'evaluation_contract_report.json')
report = json.load(open(report_path, encoding='utf-8'))
assert report['status'] == 'pass'
assert report['phase'] == 'val' and report['counts']['rows'] == 2200
assert report['counts']['candidate_pool_rows'] == 2200 * POOL_SIZE
assert report['checks']['held_out_test_accessed'] is False
assert report['checks']['cross_split_text_uid_count'] == 0
assert report['checks']['cross_split_normalized_text_count'] == 0
for name, expected in report['artifact_sha256'].items():
    state = hashlib.sha256()
    with open(os.path.join(OUTPUT, name), 'rb') as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b''):
            state.update(block)
    assert state.hexdigest() == expected, name
run_metadata = {
    'status': 'pass', 'project_commit': actual_commit, 'python': platform.python_version(),
    'dataset_index_sha256': report['source']['index_sha256'], 'phase': PHASE,
    'pool_size': POOL_SIZE, 'seed': SEED,
}
with open(os.path.join(OUTPUT, 'run_metadata.json'), 'w', encoding='utf-8') as handle:
    json.dump(run_metadata, handle, indent=2, sort_keys=True)
    handle.write('\n')
print({'dataset_root': dataset_root, **report['counts'], 'checks': report['checks']})
shutil.rmtree(WORKTREE)
print('FROZEN VALIDATION EVALUATION PROTOCOL: PASS')